# BigQuery Governance Comparison: Policy Tags vs. Direct Column Data Policies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-feb-2026/bq_legacy_tags_vs_direct_policies.ipynb)

This notebook provides a technical comparison of the **[Policy Tag (Taxonomy-based)](https://cloud.google.com/bigquery/docs/column-level-security-intro)** governance model and the **[Direct Column Data Policy (DDL-based)](https://cloud.google.com/bigquery/docs/column-data-masking-intro)** model. Both are viable strategies depending on your organizational scale and automation requirements. **Direct Column Data Policies** reached **General Availability (GA)** in February 2026. [Release Details (February 04, 2026)](https://cloud.google.com/bigquery/docs/release-notes#February_04_2026)


## Phase 1: Environment Setup & Prerequisites

We enable the BigQuery and Data Catalog APIs and initialize the project environment.

In [ ]:
%pip install google-cloud-datacatalog google-cloud-bigquery

from google.colab import auth
from google.cloud import bigquery
from google.api_core import retry, exceptions


auth.authenticate_user()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
location = 'us-central1' # @param {type:"string"}
test_user = '[user ID]' # @param {type:"string"}

# Enable the necessary APIs
!gcloud services enable datacatalog.googleapis.com bigquerydatapolicy.googleapis.com --project={project_id}

from google.cloud import bigquery
from google.cloud import datacatalog_v1

# Initialize Clients
client = bigquery.Client(project=project_id)
datacatalog = datacatalog_v1.DataCatalogClient()
# ADDED: This client is required for Taxonomies and Policy Tags
policy_manager = datacatalog_v1.PolicyTagManagerClient()

# Provision Demo Data
dataset_id = f"{project_id}.governance_comparison"
dataset = bigquery.Dataset(dataset_id)
dataset.location = location


# Delete old one if it exists in the wrong place (optional/cleaning)
client.delete_dataset(dataset_id, delete_contents=True, not_found_ok=True)
client.create_dataset(dataset, exists_ok=True)
print("Dataset Created")


table_id = f"{dataset_id}.partner_data"
schema = [
    bigquery.SchemaField("ID", "INTEGER"),
    bigquery.SchemaField("LTV", "INTEGER"),
    bigquery.SchemaField("Email", "STRING"),
]
client.create_table(bigquery.Table(table_id, schema=schema), exists_ok=True)
print("Table Created")
#client.insert_rows_json(table_id, [{"ID": 1234, "LTV": 90000, "Email": "partner_a@example.com"}])

rows_to_insert = [{"ID": 1234, "LTV": 90000, "Email": "partner_a@example.com"}]

custom_retry = retry.Retry(
    predicate=retry.if_exception_type(exceptions.NotFound, exceptions.InternalServerError),
    deadline=60,  # Will keep trying for 60 seconds total
)

print("Attempting to insert rows (with retry)...")
try:
    errors = client.insert_rows_json(
        table_id,
        rows_to_insert,
        retry=custom_retry
    )
    if not errors:
        print("Environment initialized successfully and data inserted.")
    else:
        print(f"Encountered errors during insertion: {errors}")
except Exception as e:
    print(f"Failed to insert rows even after retries: {e}")






## Phase 2: Taxonomy-based Workflow (Policy Tags)

This phase demonstrates the standard workflow for implementing column-level security using a centralized classification system. The setup typically includes the following steps:
1. **Define Taxonomy and Policy Tags**: Create a taxonomy in Data Catalog to serve as a container for regional security labels (Policy Tags).
2. **Assign Policy Tags to Columns**: Update the BigQuery table schema to associate specific columns with the defined policy tags.
3. **Enable Access Control**: Activate enforcement on the taxonomy to trigger security checks during query execution.
4. **Manage IAM Permissions**: Grant the `Fine-Grained Reader` role to authorized principals at the tag level to allow access to the raw data.

In [ ]:
import time
from google.api_core import exceptions

# Helper to find existing taxonomy by display name
def get_or_create_taxonomy(display_name, project_id, location):
    parent = f"projects/{project_id}/locations/{location}"

    # 1. Check if it already exists
    taxonomies = policy_manager.list_taxonomies(parent=parent)
    for tax in taxonomies:
        if tax.display_name == display_name:
            print(f"Found existing Taxonomy: {tax.name}")
            return tax

    # 2. If not found, create it
    print(f"Creating new Taxonomy: {display_name}")
    taxonomy_obj = datacatalog_v1.Taxonomy(
        display_name=display_name,
        activated_policy_types=[datacatalog_v1.Taxonomy.PolicyType.FINE_GRAINED_ACCESS_CONTROL]
    )
    return policy_manager.create_taxonomy(parent=parent, taxonomy=taxonomy_obj)

# Helper to find existing policy tag by display name
def get_or_create_policy_tag(parent_taxonomy_name, display_name):
    # 1. Check if it already exists
    tags = policy_manager.list_policy_tags(parent=parent_taxonomy_name)
    for tag in tags:
        if tag.display_name == display_name:
            print(f"Found existing Policy Tag: {tag.name}")
            return tag

    # 2. If not found, create it
    print(f"Creating new Policy Tag: {display_name}")
    tag_obj = datacatalog_v1.PolicyTag(display_name=display_name)
    return policy_manager.create_policy_tag(parent=parent_taxonomy_name, policy_tag=tag_obj)

# --- EXECUTION ---

# 1. Get or Create the Taxonomy and Tag
taxonomy = get_or_create_taxonomy("Legacy_Taxonomy", project_id, location)
policy_tag = get_or_create_policy_tag(taxonomy.name, "Confidential_LTV")

# 2. IAM Step: Grant access (gcloud handles existing bindings gracefully)
print(f"Ensuring IAM binding for {test_user}...")
!gcloud data-catalog taxonomies policy-tags add-iam-policy-binding {policy_tag.name} \
    --role='roles/datacatalog.categoryFineGrainedReader' \
    --member='user:{test_user}' \
    --location={location} --quiet

# 3. Attach Tag to Column via Schema Update
print("Updating BigQuery table schema...")
table = client.get_table(table_id)
new_schema = list(table.schema)
for i, field in enumerate(new_schema):
    if field.name == "LTV":
        new_field = field.to_api_repr()
        new_field["policyTags"] = {"names": [policy_tag.name]}
        new_schema[i] = bigquery.SchemaField.from_api_repr(new_field)

table.schema = new_schema
client.update_table(table, ["schema"])
print("Done! Policy Tag is active on the 'LTV' column.")

## Phase 3: DDL-based Workflow (Direct Column Data Policies)

This phase demonstrates the DDL-based approach, which allows for applying security policies directly to columns. The standard workflow for this setup includes:
1. **Create a Data Policy**: Use the `CREATE DATA POLICY` DDL statement to define a policy that includes a specific masking rule (e.g., `ALWAYS_NULL`).
2. **Grant Principal Access**: Assign permissions to the intended users or groups for that specific data policy using standard SQL `GRANT` statements.
3. **Assign Policy to Column**: Use the `ALTER TABLE` statement to bind the data policy directly to the table column. This integrates governance directly into the data definition workflow.

In [ ]:
bq_location = f"region-{location}"
policy_id = "email_sha256_mask"

# 1. Create the Direct Data Policy via DDL
ddl_create_policy = f"""
CREATE OR REPLACE DATA_POLICY `{project_id}.{bq_location}.{policy_id}`
OPTIONS (
  data_policy_type='DATA_MASKING_POLICY',
  masking_expression='ALWAYS_NULL'
)
"""
print(ddl_create_policy)
client.query(ddl_create_policy).result()
print(f"Policy '{policy_id}' created successfully in {bq_location}.")


# 2. Grant access to the user
ddl_grant = f"""
GRANT FINE_GRAINED_READ ON DATA_POLICY `{project_id}.{bq_location}.{policy_id}`
    TO \"principal://goog/subject/{test_user}\"
"""
print(f"Query '{ddl_grant}' ")
client.query(ddl_grant).result()
print(f"Policy Granted")


ddl_alter = f"""
ALTER TABLE `{table_id}`
ALTER COLUMN Email SET OPTIONS (
  data_policies = [
    '{{"name": "{project_id}.{bq_location}.{policy_id}"}}'
  ]
)
"""

print(f"Executing Query:\n{ddl_alter}")
client.query(ddl_alter).result()
print("Table Altered Successfully and Modern Direct Policy attached.")

## Phase 4: Comparative Execution & Verification

We execute a single query to see the combined effect of both governance models on **User X** (our test user).

### The Result for User X:

1. **LTV Column (Policy Tag)**: 
   - **What they see**: The **actual value** (e.g., 90000).
   - **Why**: In Phase 2, we explicitly granted User X the `FineGrainedReader` role. Without this specific role, BigQuery would have **blocked the entire query** with a 403 error.

2. **Email Column (Direct Data Policy)**: 
   - **What they see**: **NULL** (rendered as **`None`** in this Python DataFrame).
   - **Why**: In Phase 3, we granted User X access to the policy, but the policy itself defines a `ALWAYS_NULL` mask. The engine dynamically redacts the data. Even if the user is a 'Data Editor', the masking policy takes precedence for this specific column.

3. **Summary**: BigQuery governance methods are **mutually exclusive** for any single column. **Policy Tags (Taxonomy-based)** are recommended for centralized, organization-wide data classification. **Direct Column Data Policies (DDL-based)** are optimized for targeted, standalone masking rules that can be managed entirely via SQL DDL without external taxonomy dependencies.

In [ ]:
query = f"SELECT ID, LTV, Email FROM `{table_id}` LIMIT 1"
print(f"Executing query as: {test_user}")
results = client.query(query).to_dataframe()

print("\n--- MASKED RESULTS ---")
results.head()

### 5. Things to remember or know
- **Architecture Simplification**: Direct Column Data Policies (GA) remove the requirement for Data Catalog Taxonomies, allowing for standalone governance that lives closer to the data.
- **DDL-First Management**: Policies are now managed via standard SQL (`CREATE DATA POLICY`, `GRANT`), enabling 'Governance as Code' within CI/CD pipelines without needing specialized Data Catalog SDKs.
- **Dynamic Masking**: Unlike legacy tags which primarily focus on access (Allow/Deny), Direct Policies support complex masking expressions (SHA256, Nulling, Custom) directly in the DDL definition.